# Word sense disambiguation


### Setup

In [ ]:
# !pip install sentence-transformers>=2.7.0

In [ ]:
import os
import sys

import ast
import random
import numpy as np
import pandas as pd
import torch
import pickle
import matplotlib.pyplot as plt
# from tqdm import tqdm
from tqdm.auto import tqdm
from typing import List, Tuple, Optional, Union
from torch.utils.data import Dataset, DataLoader

import pickle

with open("dataset/entity_to_id.pickle", "rb") as f:
    entity_to_idx = pickle.load(f)

# wsd dataset files
TRAIN_TSV_FILE = 'dataset/train'
TEST_TSV_FILE = 'dataset/test'
EVAL_TSV_FILE = 'dataset/eval'

train_df = pd.read_csv(TRAIN_TSV_FILE, sep='\t')
test_df = pd.read_csv(TEST_TSV_FILE, sep='\t')
eval_df = pd.read_csv(EVAL_TSV_FILE, sep='\t')


train_sentences = train_df['sentence_text'].tolist()
test_sentences = test_df['sentence_text'].tolist()
eval_sentences = eval_df['sentence_text'].tolist()

train_labels = train_df['label'].apply(ast.literal_eval).tolist()
test_labels = test_df['label'].apply(ast.literal_eval).tolist()
eval_labels = eval_df['label'].apply(ast.literal_eval).tolist()

train_locations = train_df['loc'].astype(int).tolist()
test_locations = test_df['loc'].astype(int).tolist()
eval_locations = eval_df['loc'].astype(int).tolist()


train_candidates = train_df['candidates_id'].apply(ast.literal_eval).tolist()
test_candidates = test_df['candidates_id'].apply(ast.literal_eval).tolist()
eval_candidates = eval_df['candidates_id'].apply(ast.literal_eval).tolist()


device_available = torch.cuda.is_available()
device = torch.device('cuda' if device_available else 'mps' if torch.backends.mps.is_available() else 'cpu')

/datastore-xlgpu/knowdive/miniconda3/envs/pytorch/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Initialize pretrained tokenizer and model (sentence-transformers)

In [7]:
# from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModel, AutoModelForMaskedLM

from transformers import RobertaTokenizerFast

# MODEL_NAME = 'xlm-roberta-base'
# MODEL_NAME = "distilbert-base-multilingual-cased"
# MODEL_NAME = 'bert-base-uncased'
MODEL_NAME = 'roberta-base'
# MODEL_NAME = 'Qwen/Qwen3-Embedding-8B'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"device: {DEVICE}")

# model = SentenceTransformer(MODEL_NAME)

model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME)
tokenizer = RobertaTokenizerFast.from_pretrained(MODEL_NAME, use_fast=True, add_prefix_space=True)


model = model.to(device)
model.eval()
print(f"✓ Model loaded to device: {device}")
print(f"  - Model type: {model.config.model_type}")
print(f"  - Hidden size (D): {model.config.hidden_size}")
print(f"  - Number of layers: {model.config.num_hidden_layers}")
print(f"  - Number of attention heads: {model.config.num_attention_heads}")
HIDDEN_SIZE = model.config.hidden_size

print(f" vocab size: {tokenizer.vocab_size}")
print(f" max seqeunce len: {tokenizer.model_max_length}")

device: cuda
✓ Model loaded to device: cuda
  - Model type: roberta
  - Hidden size (D): 768
  - Number of layers: 12
  - Number of attention heads: 12
 vocab size: 50265
 max seqeunce len: 512


### Encoding gloss dataset in batch with sentence transformer

In [ ]:


def align_labels_with_tokens(
    encoding,
    word_labels,
    entity_to_idx,
    ignore_index=-100
):
    if isinstance(word_labels, torch.Tensor):
        word_labels = word_labels.tolist()

    word_ids = encoding.word_ids
    token_labels = [ignore_index] * len(word_ids)

    for i, word_id in enumerate(word_ids):
        if word_id is None:
            continue
        if word_id >= len(word_labels):  # skip out-of-range
            continue

        # detect LAST token of a word
        is_last_token = (
            i == len(word_ids) - 1 or
            word_ids[i + 1] != word_id
        )

        if is_last_token:
            concept_id = word_labels[word_id]
            if concept_id != ignore_index:
                key = str(concept_id)
                token_labels[i] = entity_to_idx.get(key, ignore_index)

    return token_labels


def collate_fn(batch):
    sentences, labels, locs = zip(*batch)
    return list(sentences), list(labels), list(locs)

# Define a PyTorch Dataset for sentences
class SentenceDataset(Dataset):
    """
    Simple dataset wrapper for sentences.

    Returns a sentence string at each index.
    Shape: Returns string (will be tokenized in DataLoader or encoding function)
    """
    def __init__(self, sentences: List[str], labels: List[int], locs: List[List[int]]):
        self.sentences = sentences
        self.labels = labels
        # self.candidates = []
        self.locs = locs
    
    def __len__(self) -> int:
        return len(self.sentences)
    
    def __getitem__(self, idx: int) -> str:
        # return self.sentences[idx]
        # return self.labels[idx]
        return self.sentences[idx], self.labels[idx], self.locs[idx]# self.candidates[idx]




def encode_dataset(
    sentences,
    labels,
    locations,
    tokenizer,
    model,
    device,
    batch_size=32,
    max_length=128,
    show_progress=True
):
    dataset = SentenceDataset(sentences, labels, locs=locations)

    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        collate_fn=collate_fn,
        shuffle=False,
        num_workers=0
    )

    all_embeddings = []
    all_token_labels = []
    total_processed = 0

    iterator = tqdm(dataloader, desc="Encoding", disable=not show_progress)

    for batch in iterator:
        batch_sentences, batch_word_labels, batch_locs = batch
        batch_words = [s.split() for s in batch_sentences]
        batch_size_actual = len(batch_words)

        # Tokenize
        encodings = tokenizer(
            batch_words,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
            is_split_into_words=True,
            # add_prefix_space=True, #roberta-based
        )

        inputs = {k: v.to(device) for k, v in encodings.items()}

        # Align labels
        batch_labels = []
        for i in range(batch_size_actual):
            token_labels = align_labels_with_tokens(
                encodings.encodings[i],
                batch_word_labels[i],
                entity_to_idx
            )
            batch_labels.append(token_labels)

        # Pad labels
        padded_labels = []
        for lbl in batch_labels:
            if len(lbl) < max_length:
                lbl = lbl + [-100] * (max_length - len(lbl))
            else:
                lbl = lbl[:max_length]
            padded_labels.append(lbl)

        batch_labels = torch.tensor(padded_labels, dtype=torch.long)
        all_token_labels.append(batch_labels.numpy())

        # Move inputs to device
        # inputs = {k: v.to(device) for k, v in inputs.items()}


        # Forward
        with torch.no_grad():
            outputs = model(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                output_hidden_states=True,
                return_dict=True
            )

        hidden_states = outputs.hidden_states[-1]  # (B, T, H)
        batch_embeddings = []

        # Extract target word embeddings
        for i in range(batch_size_actual):
            word_ids = encodings.encodings[i].word_ids
            target_word_id = batch_locs[i]

            token_idxs = [
                j for j, w_id in enumerate(word_ids)
                if w_id == target_word_id
            ]

            if not token_idxs:
                emb = hidden_states[i, 0]  # CLS fallback
            else:
                emb = hidden_states[i, token_idxs[-1]]  # last subword

            batch_embeddings.append(emb)

        batch_embeddings = torch.stack(batch_embeddings)
        all_embeddings.append(batch_embeddings.cpu().numpy())

        total_processed += batch_size_actual
        iterator.set_description(f"Encoded {total_processed}/{len(dataset)}")

    final_embeddings = np.vstack(all_embeddings)
    final_token_labels = np.vstack(all_token_labels)

    return final_embeddings, total_processed, final_token_labels


print("\n" + "=" * 60)
print("Starting full dataset encoding...")
print("=" * 60)

# all_sentences = eval_sentences# + train_sentences + test_sentences + eval_sentences
# all_labels = eval_labels# + train_labels + test_labels + eval_labels
# Encode all sentences
# all_embeddings, processed_count, all_token_labels = encode_dataset(
#     sentences=all_sentences,
#     labels=all_labels,
#     tokenizer=tokenizer,
#     model=model,
#     device=device,
#     # batch_size=1,
#     batch_size=32,
#     max_length=128,
#     show_progress=True
# )

train_embeddings, processed_count, train_token_labels = encode_dataset(
    sentences=train_sentences,
    labels=train_labels,
    locations=train_locations,
    tokenizer=tokenizer,
    model=model,
    device=device,
    batch_size=32,
    max_length=128,
    show_progress=True
)


test_embeddings, processed_count, test_token_labels = encode_dataset(
    sentences=test_sentences,
    labels=test_labels,
    locations=test_locations,
    tokenizer=tokenizer,
    model=model,
    device=device,
    batch_size=32,
    max_length=128,
    show_progress=True
)

eval_embeddings, processed_count, eval_token_labels = encode_dataset(
    sentences=eval_sentences,
    labels=eval_labels,
    locations=eval_locations,
    tokenizer=tokenizer,
    model=model,
    device=device,
    batch_size=32,
    max_length=128,
    show_progress=True
)

all_embeddings = np.vstack([train_embeddings, test_embeddings, eval_embeddings])



Starting full dataset encoding...


Encoded 424/424: 100%|██████████| 14/14 [00:00<00:00, 16.50it/s]


### Normalizing and processing embeddings

We're keeping both raw and normalized embedding for the further testing modes.

Moreover, saving the embeddings for the convenience of later direct use.

In [ ]:
# L2-normalize embeddings to unit-length vectors, useful for cosine similarity
def l2_normalize(embeddings: np.ndarray) -> np.ndarray:
    """
    L2-normalize embeddings to unit vectors.
    """
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)  # (N, 1)
    norms = np.maximum(norms, 1e-12)
    normalized = embeddings / norms  # (N, D)
    
    return normalized

train_embeddings_raw = train_embeddings.copy()
train_token_labels_to_save = train_token_labels.copy()

test_embeddings_raw = test_embeddings.copy()
test_token_labels_to_save = test_token_labels.copy()

eval_embeddings_raw = eval_embeddings.copy()
eval_token_labels_to_save = eval_token_labels.copy()

train_embeddings_normalized = l2_normalize(train_embeddings_raw)
test_embeddings_normalized = l2_normalize(test_embeddings_raw)
eval_embeddings_normalized = l2_normalize(eval_embeddings_raw)

train_output_file = 'sentence_embeddings_labels_loc_roberta_base.npz'

np.savez_compressed(
    train_output_file,
    train_embeddings=train_embeddings_normalized,
    train_labels=train_token_labels_to_save,
    test_embeddings=test_embeddings_normalized,
    test_labels=test_token_labels_to_save,
    eval_embeddings=eval_embeddings_normalized,
    eval_labels=eval_token_labels_to_save
)


### Load embeddings

In [ ]:
output_file = 'sentence_embeddings_labels.npz'
loaded_data = np.load(output_file)

loaded_train_embeddings = loaded_data['train_embeddings']
loaded_train_labels = loaded_data['train_labels']
loaded_test_embeddings = loaded_data['test_embeddings'] 
loaded_test_labels = loaded_data['test_labels']
loaded_eval_embeddings = loaded_data['eval_embeddings']
loaded_eval_labels = loaded_data['eval_labels']
